<a href="https://colab.research.google.com/github/nouramar228-cyber/intermediate-ml-practice-solutions/blob/main/Feature_Engineering_Exercise_(Core).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment:**

In this exercise, you will be working with data about bike share rentals. [You can download the data here.](https://drive.google.com/file/d/1Sl6mHRE9rk3n5b0DcImv4-F-Cqr075GH/view?usp=sharing)

Your task is to engineer some new features to try to improve a model's ability to predict the total number of bike share rentals during a given hour of the day.

1. Import the data the drop the 'casual' and 'registered' columns. These are redundant with your target, 'count'.
2. Transform the 'datetime' column into a datetime type and use it to create 3 new columns in the data frame containing the:
   * 1. Name of the Month
   * 2. Name of the Day of the Week
   * 3. Hour of the Day
     * 1. Make sure all 3 new columns are 'object' datatype so they can be one-hot encoded later.
     * 2. Drop the 'datetime' and 'season' columns. These are now redundant.
3. The temperatures in the 'temp' and 'atemp' columns are in Celsius. Use `.apply()` and a Lambda function to convert them to Fahrenheit.

4. Create a new column, 'temp_variance,' which shows how much warmer or colder the current temperature ('temp') is than the average temperate for that day of the year ('atemp'). If the current temperature is warmer than average ('atemp'), the value in 'temp_variance' should be positive.
Drop the 'atemp' column.




1. Import and load data

In [ ]:
import pandas as pd
# load the data
fpath = '/content/drive/MyDrive/AXSOSACADEMY/AXSOSACADEMY/05-IntermediateML/Week17/Data/bikeshare_train - bikeshare_train.csv'
df = pd.read_csv(fpath)
df.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 0:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 1:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 2:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 3:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 4:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [ ]:
df = df.drop(columns=['casual','registered'])


2. Transform the 'datetime' column into a datetime type and use it to create 3 new columns in the data frame

In [ ]:
#Convert datetime to Datetime Type
df['datetime'] = pd.to_datetime(df['datetime'])

In [ ]:
df['month_name'] = df['datetime'].dt.month_name().astype('object')
df['day_of_week'] = df['datetime'].dt.day_name().astype('object')
df['hour'] = df['datetime'].dt.hour.astype('object')

In [ ]:
# Drop Redundant Columns
df = df.drop(columns=['datetime', 'season'])

In [ ]:
# Convert Celsius Temperatures to Fahrenheit
df['temp'] = df['temp'].apply(lambda c: (c * 9/5) + 32)

df['atemp'] = df['atemp'].apply(lambda c: (c * 9/5) + 32)

In [ ]:
# Create the temp_variance Feature
df['temp_variance'] = df['temp'] - df['atemp']
# Positive values mean the current temperature is warmer than average.
# Negative values mean the current temperature is colder than average.

In [ ]:
# Drop the atemp Column
df = df.drop(columns=['atemp'])

In [ ]:
df.head()


,holiday,workingday,weather,temp,humidity,windspeed,count,month_name,day_of_week,hour,temp_variance
0,0,0,1,49.712,81,0.0,16,January,Saturday,0,-8.199
1,0,0,1,48.236,80,0.0,40,January,Saturday,1,-8.307
2,0,0,1,48.236,80,0.0,32,January,Saturday,2,-8.307
3,0,0,1,49.712,75,0.0,13,January,Saturday,3,-8.199
4,0,0,1,49.712,75,0.0,1,January,Saturday,4,-8.199


In [ ]:
df.dtypes

,0
holiday,int64
workingday,int64
weather,int64
temp,float64
humidity,int64
windspeed,float64
count,int64
month_name,object
day_of_week,object
hour,object


**Original Feature Set vs Engineered Feature Set**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Original Dataset
df = pd.read_csv(fpath)
original_df = df.copy()
original_df = original_df.drop(columns=['casual', 'registered'])

In [ ]:
# Engineered Dataset
engineered_df = df.copy()
engineered_df = engineered_df.drop(columns=['casual', 'registered'])

In [ ]:
engineered_df['datetime'] = pd.to_datetime(engineered_df['datetime'])
engineered_df['month_name'] = engineered_df['datetime'].dt.month_name().astype('object')
engineered_df['day_of_week'] = engineered_df['datetime'].dt.day_name().astype('object')
engineered_df['hour'] = engineered_df['datetime'].dt.hour.astype('object')

In [ ]:
engineered_df = engineered_df.drop(columns=['datetime', 'season'])


In [ ]:
engineered_df['temp'] = engineered_df['temp'].apply(lambda c: (c * 9/5) + 32)
engineered_df['atemp'] = engineered_df['atemp'].apply(lambda c: (c * 9/5) + 32)

In [ ]:
engineered_df['temp_variance'] = engineered_df['temp'] - engineered_df['atemp']

engineered_df = engineered_df.drop(columns=['atemp'])


In [ ]:
def evaluate_model(df):
    X = df.drop(columns=['count'])
    y = df['count']

    categorical_cols = X.select_dtypes(include=['object']).columns

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        ],
        remainder='passthrough'
    )

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    return mae, mse, rmse, r2

In [ ]:
# Evaluate both datasets
original_mae, original_mse, original_rmse, original_r2 = evaluate_model(original_df)


In [ ]:
print("Original Dataset")
print("MAE:", original_mae)
print("MSE:", original_mse)
print("RMSE:", original_rmse)
print("R2:", original_r2)

Original Dataset
MAE: 101.12123507805326
MSE: 24524.49413530762
RMSE: 156.60298252366596
R2: 0.25698967527349825


In [ ]:
engineered_mae, engineered_mse, engineered_rmse, engineered_r2 = evaluate_model(engineered_df)

In [ ]:
print("\nEngineered Dataset")
print("MAE:", engineered_mae)
print("MSE:", engineered_mse)
print("RMSE:", engineered_rmse)
print("R2:", engineered_r2)


Engineered Dataset
MAE: 48.18091222834405
MSE: 5242.230883583728
RMSE: 72.40325188542106
R2: 0.8411778995475706


MAE	for Original Features: 	101.12, R² Score: 0.257

MAE Engineered Features	48.18, 	R² Score: 0.841

The engineered feature set significantly improved the model’s predictive performance.

The engineered features reduced prediction error substantially and increased the model’s ability to explain variance in bike rental counts.